# Treinamento da U-Net no Colab (GPU T4) — 6 classes

Treina a U-Net do GeoRural com **leave-one-tile-out** (validação em bioma inteiro) e **pesos de classe**.

## Antes de rodar
1. Envie para o Colab o `georural.zip` (repositório) e o `unet_dataset.zip` (conteúdo de `data/processed/unet/`).
2. Ative a GPU: *Runtime > Change runtime type > T4 GPU*.
3. Rode as células em ordem.

Ao final, baixe `models/brazil_unet.pth` e coloque em `models/` no projeto.

In [ ]:
import os, sys, zipfile, glob

# from google.colab import files
# files.upload()

REPO_ZIP = 'georural.zip'
DATA_ZIP = 'unet_dataset.zip'

if os.path.exists(REPO_ZIP) and not os.path.isdir('GeoRural'):
    with zipfile.ZipFile(REPO_ZIP) as z:
        z.extractall('.')

if os.path.isdir('GeoRural'):
    os.chdir('GeoRural')

os.makedirs('data/processed/unet', exist_ok=True)
if os.path.exists(DATA_ZIP):
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall('data/processed/unet')

sys.path.insert(0, os.getcwd())
print('Projeto em:', os.getcwd())
print(sorted(glob.glob('data/processed/unet/*_image.npy')))

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: GPU não ativa. Ative Runtime > Change runtime type > T4 GPU.')

In [ ]:
import numpy as np
from src.classifiers.unet_cnn import (
    build_unet, leave_one_tile_out, load_unet_tiles, train_unet,
)
from src.config import IGNORE_LABEL, NUM_CLASSES

tiles = load_unet_tiles('data/processed/unet')
print('Tiles:', list(tiles))

VAL_TILE = 'unet_piracicaba' if 'unet_piracicaba' in tiles else next(iter(tiles))
print('Validação:', VAL_TILE)

X_train, y_train, X_val, y_val = leave_one_tile_out(
    tiles, VAL_TILE, patch_size=64, stride=32
)
print('Treino:', X_train.shape, 'Validação:', X_val.shape)

valid = y_train != IGNORE_LABEL
counts = np.bincount(
    y_train[valid].astype(int).ravel(), minlength=NUM_CLASSES
)[:NUM_CLASSES].astype(float)
counts[counts == 0] = 1.0
class_weights = counts.sum() / (counts * len(counts))
print('Pesos:', np.round(class_weights, 4).tolist())

In [ ]:
import os
os.makedirs('models', exist_ok=True)

model = build_unet(
    input_shape=(64, 64, X_train.shape[-1]), num_classes=NUM_CLASSES
)
history = train_unet(
    model, X_train, y_train, X_val, y_val,
    epochs=100, batch_size=16, learning_rate=0.001,
    save_path='models/brazil_unet.pth',
    class_weights=class_weights,
    ignore_index=IGNORE_LABEL,
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['val_acc'], label='Val')
plt.title('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

from google.colab import files
files.download('models/brazil_unet.pth')